# Multiple-Choice Recognition Analysis

Figures and summary statistics for one MC variant (set `VARIANT` below):

| Figure | Content | Used in the paper |
|---|---|---|
| fig0 / fig0b | author-count distribution (overall / per field) | |
| fig1a / fig1b / fig1c | chance-corrected MC accuracy by citation bin (overall / per field / per paper type) | |
| fig2a | recall-recognition gap under HR1-HR5 | Appendix, gap under all HR variants |
| fig2d | recall accuracy, recognition accuracy and gap | Section 6, main MC figure |
| fig2e | binary perfect recall vs. recognition | Appendix, binary perfect recall |
| fig2b / fig2c | gap per field, recall accuracy per HR variant | |
| fig4a / fig4b | residual recognition among recall failures | |
| fig5a | position-bias check | |
| fig_k_vs_accuracy | accuracy vs. number of authors | |

**Input:** `results/mc_recognition/<variant>/results.csv`, `bin_summary.csv` (from `evaluate.py`).
**Output:** PDF and PNG figures in `results/mc_recognition/<variant>/figures/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sys.path.insert(0, str(Path.cwd()))  # run this notebook from its own folder
import common
from common import BIN_ORDER, config

VARIANT = "original"  # "original", "same_field_swap" or "collaborator_swap"

paths = common.variant_paths(VARIANT)
FIG_DIR = paths.figures_dir
FIG_DIR.mkdir(parents=True, exist_ok=True)
FIELDS = sorted(config.FIELDS)

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = sns.color_palette("tab10")


def save(fig_name: str, **kwargs) -> None:
    """Save the current figure as PDF and PNG."""
    plt.savefig(FIG_DIR / f"{fig_name}.pdf", **kwargs)
    plt.savefig(FIG_DIR / f"{fig_name}.png", dpi=150, **{k: v for k, v in kwargs.items() if k != "dpi"})


results = pd.read_csv(paths.results_csv)
summary = pd.read_csv(paths.bin_summary_csv)
summary["citation_bin"] = pd.Categorical(summary["citation_bin"], categories=BIN_ORDER, ordered=True)
summary = summary.sort_values("citation_bin")
results["citation_bin"] = pd.Categorical(results["citation_bin"], categories=BIN_ORDER, ordered=True)

overall = summary[(summary["field"] == "all") & (summary["paper_type"] == "all")].copy()
print(f"Variant '{VARIANT}': {len(results)} per-paper results, {len(summary)} bin-summary rows")
overall[["citation_bin", "n", "acc_mc", "acc_mc_cc", "gap_hr2", "residual_rate_50"]]

## n_authors Distribution

In [ ]:
nauth_counts = results["n_authors"].value_counts().sort_index()
nauth_df = pd.DataFrame({
    "n_authors": nauth_counts.index,
    "n_papers": nauth_counts.values,
    "pct": (nauth_counts.values / len(results) * 100).round(2),
})
print("Author count distribution:")
print(nauth_df.to_string(index=False))
print(f"\nMedian n_authors: {results['n_authors'].median():.1f}")
print(f"Mean n_authors:   {results['n_authors'].mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(nauth_counts.index, nauth_counts.values, color=PALETTE[0], width=0.75)
ax.set_xlabel("Number of Authors (k)")
ax.set_ylabel("Number of Papers")
ax.set_title("n_authors Distribution (evaluation set)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for k, v in zip(nauth_counts.index, nauth_counts.values):
    if v > 100:
        ax.text(k, v + 15, str(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save("fig0_nauthors_dist", dpi=150)
plt.show()

## n_authors Distribution by Field

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, field in enumerate(FIELDS):
    ax = axes[i]
    field_data = results[results["field"] == field]["n_authors"]
    counts = field_data.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=PALETTE[i % len(PALETTE)], width=0.75)
    ax.set_title(field.title(), fontsize=12)
    ax.set_xlabel("k (authors)", fontsize=9)
    ax.set_ylabel("Papers", fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    med = field_data.median()
    mean = field_data.mean()
    ax.axvline(med, color="black", linestyle="--", linewidth=1, label=f"Median={med:.0f}")
    ax.legend(fontsize=8)
    ax.text(0.97, 0.95, f"mean={mean:.1f}", transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="gray")

fig.suptitle("n_authors Distribution by Field", fontsize=15, y=1.01)
plt.tight_layout()
save("fig0b_nauthors_by_field", dpi=150, bbox_inches="tight")
plt.show()

## Fig 1-A — MC-Multi Accuracy by Citation Bin (Overall)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(overall))
ax.plot(x, overall["acc_mc_cc"], marker="o", color=PALETTE[0], label="MC (chance-corrected)")
ax.fill_between(x, overall["acc_mc_cc_ci95_lo"], overall["acc_mc_cc_ci95_hi"],
                alpha=0.2, color=PALETTE[0])
ax.axhline(0, color="black", linestyle="--", linewidth=1, label="Chance (0)")

ax.set_xticks(list(x))
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right")
ax.set_xlabel("Citation Bin")
ax.set_ylabel("Chance-Corrected Accuracy")
ax.set_title("Fig 1-A: MC Accuracy by Citation Bin (Overall)")
ax.legend()
plt.tight_layout()
save("fig1a_overall_accuracy", dpi=150)
plt.show()

## Fig 1-B — MC-Multi Accuracy by Citation Bin (Per Field)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for i, field in enumerate(FIELDS):
    ax = axes[i]
    grp = summary[(summary["field"] == field) & (summary["paper_type"] == "all")].copy()
    grp = grp.sort_values("citation_bin")
    x = range(len(grp))
    ax.plot(x, grp["acc_mc_cc"], marker="o", color=PALETTE[i % len(PALETTE)])
    ax.fill_between(x, grp["acc_mc_cc_ci95_lo"], grp["acc_mc_cc_ci95_hi"],
                    alpha=0.2, color=PALETTE[i % len(PALETTE)])
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(field.title())
    ax.set_xticks(list(x))
    ax.set_xticklabels(grp["citation_bin"].tolist(), rotation=60, ha="right", fontsize=7)
    if i % 4 == 0:
        ax.set_ylabel("Chance-Corrected Acc.")

fig.suptitle("Fig 1-B: MC Accuracy by Field", fontsize=14)
plt.tight_layout()
save("fig1b_field_accuracy", dpi=150)
plt.show()

## Fig 1-C — MC-Multi Accuracy by Paper Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for i, ptype in enumerate(["single", "multi"]):
    ax = axes[i]
    grp = summary[(summary["paper_type"] == ptype) & (summary["field"] == "all")].copy()
    grp = grp.sort_values("citation_bin")
    x = range(len(grp))
    ax.plot(x, grp["acc_mc_cc"], marker="o", color=PALETTE[i])
    ax.fill_between(x, grp["acc_mc_cc_ci95_lo"], grp["acc_mc_cc_ci95_hi"],
                    alpha=0.2, color=PALETTE[i])
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    n_total = int(summary[(summary["paper_type"] == ptype) & (summary["field"] == "all")]["n"].sum())
    ax.set_title(f"{ptype.title()}-Author Papers (n={n_total})")
    ax.set_xticks(list(x))
    ax.set_xticklabels(grp["citation_bin"].tolist(), rotation=45, ha="right")
    ax.set_xlabel("Citation Bin")
    if i == 0:
        ax.set_ylabel("Chance-Corrected Accuracy")

fig.suptitle("Fig 1-C: MC Accuracy by Paper Type", fontsize=14)
plt.tight_layout()
save("fig1c_papertype_accuracy", dpi=150)
plt.show()

## Fig 2-A — Recall–Recognition Gap (Overall, HR2-based)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

x = range(len(overall))

# --- Only left and bottom spines, in black ---
ax.spines["left"].set_color("black")
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_color("black")
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.plot(x, overall["gap_hr1"], marker="s", linestyle="--", color=PALETTE[3], alpha=0.7, label="Gap (HR1)")
ax.plot(x, overall["gap_hr2"], marker="o", color=PALETTE[2], label="Gap (HR2)")
ax.plot(x, overall["gap_hr3"], marker="^", linestyle=":", color=PALETTE[4], alpha=0.7, label="Gap (HR3)")
ax.plot(x, overall["gap_hr4"], marker="D", linestyle="-.", color=PALETTE[5], alpha=0.85, label="Gap (HR4)")
ax.plot(x, overall["gap_hr5"], marker="v", linestyle=(0, (3,1,1,1)), color=PALETTE[6], alpha=0.85, label="Gap (HR5)")

ax.set_xticks(list(x))
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right")
ax.tick_params(axis="y", labelsize=13, colors="black")
ax.tick_params(axis="x", labelsize=13, colors="black")
ax.set_xlabel("Citation Bin", fontsize=15, color="black", labelpad=8)
ax.set_ylabel("Recognition − Recall Gap", fontsize=15, color="black", labelpad=8)
ax.legend(fontsize=9)
plt.tight_layout()
save("fig2a_overall_gap", dpi=150)
plt.show()

## Fig 2-D — Recall vs. Recognition Accuracy (with Gap Context)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

x = np.arange(len(overall))

# --- Gap HR2 only (green) ---
ax.plot(x, overall["gap_hr2"], marker="s", markersize=5, linestyle="-",
        color="#2ca02c", linewidth=1.8, alpha=0.9, label="Gap HR2 (Recognition − Recall)", zorder=2)

# --- Recognition Accuracy (blue, bold, no CI) ---
blue = "#2166ac"
ax.plot(x, overall["acc_mc_cc"], marker="o", markersize=7, color=blue,
        linewidth=2.8, label="Recognition Accuracy", zorder=5)

# --- Recall Accuracy 1−HR2 (orange, bold) ---
orange = "#d95f02"
ax.plot(x, overall["acc_recall_hr2"], marker="s", markersize=7, color=orange,
        linewidth=2.8, label="Recall Accuracy (1−HR2)", zorder=5)

# --- Axis limits & ticks ---
ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right", fontsize=13, color="black")
ax.tick_params(axis="y", labelsize=13, colors="black")
ax.tick_params(axis="x", colors="black")

# --- Axis labels ---
ax.set_xlabel("Citation Count", fontsize=15, color="black", labelpad=8)
ax.set_ylabel("Accuracy / Gap", fontsize=15, color="black", labelpad=8)

# --- Only left and bottom spines, in black ---
ax.spines["left"].set_color("black")
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_color("black")
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Full grid (horizontal and vertical) ---
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#e0e0e0", linewidth=0.8)
ax.xaxis.grid(True, color="#e0e0e0", linewidth=0.8)

# --- Legend ---
ax.legend(fontsize=9, loc="lower right", framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
save("fig2d_recall_recognition_gap", bbox_inches="tight")
plt.show()
print("Saved fig2d_recall_recognition_gap.pdf / .png")

## Fig 2-E — Perfect-Recall vs. MC Recognition (Binary Metric Comparison)

Uses binary perfect-recall (correct only when all author names match exactly, H=0 and U=0)
instead of the partial-credit 1−HR₂, for metric-commensurability robustness check.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))

x = np.arange(len(overall))

blue   = "#2166ac"
orange = "#d95f02"
green  = "#2ca02c"
grey   = "#888888"

# Recognition accuracy (binary MC)
ax.plot(x, overall["acc_mc_cc"], marker="o", markersize=7, color=blue,
        linewidth=2.8, label="Recognition accuracy (MC, binary)", zorder=5)

# Binary perfect-recall (H=0, U=0)
ax.plot(x, overall["acc_recall_perfect"], marker="^", markersize=7, color=orange,
        linewidth=2.8, label="Perfect-recall accuracy (binary, H=U=0)", zorder=5)

# Partial-credit 1−HR₂ (greyed out, for reference)
ax.plot(x, overall["acc_recall_hr2"], marker="s", markersize=5, color=grey,
        linewidth=1.5, linestyle="--", alpha=0.6, label="Recall accuracy (1−HR2, partial-credit)", zorder=3)

# Gap under binary perfect-recall
ax.plot(x, overall["gap_perfect"], marker="D", markersize=5, color=green,
        linewidth=1.8, linestyle="-", alpha=0.9, label="Gap (Recognition − Perfect-recall)", zorder=2)

ax.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right", fontsize=13, color="black")
ax.tick_params(axis="y", labelsize=13, colors="black")
ax.tick_params(axis="x", colors="black")
ax.set_xlabel("Citation Count", fontsize=15, color="black", labelpad=8)
ax.set_ylabel("Accuracy / Gap", fontsize=15, color="black", labelpad=8)

ax.spines["left"].set_color("black"); ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_color("black"); ax.spines["bottom"].set_linewidth(1.2)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.set_axisbelow(True)
ax.yaxis.grid(True, color="#e0e0e0", linewidth=0.8)
ax.xaxis.grid(True, color="#e0e0e0", linewidth=0.8)

ax.legend(fontsize=9, loc="center left", bbox_to_anchor=(0.02, 0.5), framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
save("fig2e_perfect_recall_gap", bbox_inches="tight")
plt.show()

# Print numbers for paper
print("\nBin | Perf-recall | MC (CC) | Gap (binary)")
for _, row in overall.iterrows():
    print(f"{row['citation_bin']:12s} | {row['acc_recall_perfect']:.3f}       | {row['acc_mc_cc']:.3f}    | {row['gap_perfect']:.3f}")

## Fig 2-B — Recall–Recognition Gap by Field (HR2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for i, field in enumerate(FIELDS):
    ax = axes[i]
    grp = summary[(summary["field"] == field) & (summary["paper_type"] == "all")].copy()
    grp = grp.sort_values("citation_bin")
    x = range(len(grp))
    ax.plot(x, grp["gap_hr2"], marker="o", color=PALETTE[i % len(PALETTE)])
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(field.title())
    ax.set_xticks(list(x))
    ax.set_xticklabels(grp["citation_bin"].tolist(), rotation=60, ha="right", fontsize=7)
    if i % 4 == 0:
        ax.set_ylabel("Recognition − Recall Gap")

fig.suptitle("Fig 2-B: Recall–Recognition Gap by Field (HR2)", fontsize=14)
plt.tight_layout()
save("fig2b_field_gap", dpi=150)
plt.show()

## Fig 2-C — HR1–HR5 Absolute Recall Rates by Citation Bin

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(overall))
hr_specs = [
    ("acc_recall_hr1", "HR1 (precision-based)",    PALETTE[0], "o",  "-"),
    ("acc_recall_hr2", "HR2 (combined)",            PALETTE[1], "s",  "--"),
    ("acc_recall_hr3", "HR3 (F1-based)",            PALETTE[2], "^",  ":"),
    ("acc_recall_hr4", "HR4 (hallucination-only)",  PALETTE[5], "D",  "-."),
    ("acc_recall_hr5", "HR5 (omission-only)",       PALETTE[6], "v",  (0, (3,1,1,1))),
]

for col, label, color, marker, ls in hr_specs:
    ax.plot(x, overall[col], marker=marker, linestyle=ls, color=color, label=label, alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right")
ax.set_xlabel("Citation Bin")
ax.set_ylabel("Recall Accuracy (1 − HR)")
ax.set_title("Fig 2-C: Recall Accuracy by HR Variant and Citation Bin")
ax.legend(fontsize=9)
plt.tight_layout()
save("fig2c_hr_variants_by_bin", dpi=150)
plt.show()

print("\nOverall recall accuracies at lowest bin (0-2):")
low_bin = overall[overall["citation_bin"] == "0-2"].iloc[0]
for col, label, *_ in hr_specs:
    print(f"  {label}: {low_bin[col]:.4f}")
print("\nOverall recall accuracies at highest bin (4097+):")
high_bin = overall[overall["citation_bin"] == "4097+"].iloc[0]
for col, label, *_ in hr_specs:
    print(f"  {label}: {high_bin[col]:.4f}")

## Fig 4-A — Residual Knowledge Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(overall))
ax.plot(x, overall["residual_rate_50"], marker="o", color=PALETTE[5],
        label="Residual rate (HR2 > 0.5)")
ax.plot(x, overall["residual_rate_75"], marker="s", linestyle="--", color=PALETTE[6], alpha=0.8,
        label="Residual rate (HR2 > 0.75, sensitivity)")
ax.axhline(0.25, color="gray", linestyle=":", linewidth=1, label="Chance (25%)")

ax.set_xticks(list(x))
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right")
ax.set_xlabel("Citation Bin")
ax.set_ylabel("Residual Rate (Recognition | Recall Fails)")
ax.set_title("Fig 4-A: Residual Knowledge Rate by Citation Bin")
ax.legend()
plt.tight_layout()
save("fig4a_residual_rate", dpi=150)
plt.show()

## Fig 4-B — Mean HR2 by Recognition Outcome

In [ ]:
hr2_by_outcome = (
    results[results["hr2"] > 0.5]
    .groupby(["citation_bin", "correct"], observed=False)["hr2"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
for outcome, label, color in [
    (1, "Recognition Success", PALETTE[0]),
    (0, "Recognition Failure", PALETTE[1]),
]:
    sub = hr2_by_outcome[hr2_by_outcome["correct"] == outcome].sort_values("citation_bin")
    x_vals = [BIN_ORDER.index(b) for b in sub["citation_bin"]]
    ax.plot(x_vals, sub["hr2"].tolist(), marker="o", color=color, label=label)

ax.set_xticks(range(len(BIN_ORDER)))
ax.set_xticklabels(BIN_ORDER, rotation=45, ha="right")
ax.set_xlabel("Citation Bin")
ax.set_ylabel("Mean HR2 (among recall-fail papers)")
ax.set_title("Fig 4-B: Mean HR2 by Recognition Outcome (Recall-Fail Subset)")
ax.legend()
plt.tight_layout()
save("fig4b_hr2_by_outcome", dpi=150)
plt.show()

## Fig 5-A — Position Bias Check

In [ ]:
positions = ["A", "B", "C", "D"]
correct_pos_dist = results["correct_pos"].value_counts(normalize=True).reindex(positions)
chosen_pos_dist  = results["parsed_letter"].value_counts(normalize=True).reindex(positions)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(positions))
width = 0.35

ax.bar(x - width/2, correct_pos_dist.values, width,
       label="Correct answer placement", color=PALETTE[0])
ax.bar(x + width/2, chosen_pos_dist.fillna(0).values, width,
       label="Model's chosen position", color=PALETTE[1])
ax.axhline(0.25, color="black", linestyle="--", linewidth=1, label="Expected (25%)")

ax.set_xticks(x)
ax.set_xticklabels(positions)
ax.set_xlabel("Choice Position")
ax.set_ylabel("Frequency")
ax.set_title("Fig 5-A: Position Bias Check ")
ax.legend()

max_dev = (chosen_pos_dist.fillna(0) - 0.25).abs().max()
ax.text(0.98, 0.02, f"Max bias: {max_dev:.3f}", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=10, color="red")

plt.tight_layout()
save("fig5a_position_bias", dpi=150)
plt.show()

print("Correct pos distribution:", correct_pos_dist.round(4).to_dict())
print("Chosen pos distribution: ", chosen_pos_dist.fillna(0).round(4).to_dict())
print(f"Max deviation from chance: {max_dev:.4f}")
if max_dev > 0.05:
    print("WARNING: Position bias > 5 pp — consider applying correction.")
else:
    print("OK: No significant position bias (< 5 pp).")

## n_authors vs Accuracy

In [ ]:
orig = pd.read_csv(config.EVAL_SET_CSV, usecols=["paperId", "citationCount"])
results_cite = results.merge(orig, on="paperId", how="left")

# Per-k exact grouping (k=1..8 individually, k>=9 grouped)
def k_label(k: int) -> str:
    return "≥9" if k >= 9 else str(k)

results_cite["k_group"] = results_cite["n_authors"].apply(k_label)
K_ORDER = ["1", "2", "3", "4", "5", "6", "7", "8", "≥9"]

k_acc = (
    results_cite.groupby("k_group", observed=True)
    .agg(
        n=("correct", "count"),
        n_correct=("correct", "sum"),
        mean_citations=("citationCount", "mean"),
    )
    .reindex(K_ORDER)
)
k_acc["acc_mc"] = k_acc["n_correct"] / k_acc["n"]
k_acc["acc_mc_cc"] = (k_acc["acc_mc"] - 0.25) / 0.75
k_acc["mean_citations"] = k_acc["mean_citations"].round(1)

print("k vs accuracy (per-k exact grouping) with mean citations:")
print(k_acc[["n", "acc_mc_cc", "mean_citations"]].round(4).to_string())

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bars = ax1.bar(range(len(K_ORDER)), k_acc["acc_mc_cc"], color=PALETTE[0], alpha=0.85, label="MC acc cc")
ax2.plot(range(len(K_ORDER)), k_acc["mean_citations"], marker="o", color=PALETTE[1],
         linewidth=2, label="Mean citations", zorder=3)

ax1.axhline(0, color="black", linestyle="--", linewidth=1)
ax1.set_xticks(range(len(K_ORDER)))
ax1.set_xticklabels(K_ORDER)
ax1.set_xlabel("Number of Authors (k)")
ax1.set_ylabel("Chance-Corrected Accuracy", color=PALETTE[0])
ax2.set_ylabel("Mean Citation Count", color=PALETTE[1])
ax1.set_title("MC Accuracy and Mean Citations vs Author Count (k)")

for i, row in enumerate(k_acc.itertuples()):
    ax1.text(i, row.acc_mc_cc + 0.01, f"n={int(row.n)}", ha="center", va="bottom", fontsize=8)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc="lower right")

plt.tight_layout()
save("fig_k_vs_accuracy", dpi=150)
plt.show()

rho, p = stats.spearmanr(results["n_authors"], results["correct"])
print(f"\nSpearman ρ(n_authors, correct) = {rho:.4f}, p = {p:.4g}")